In [ ]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## input

In [ ]:
from src.preprocessing import pp,gene_pair_split
from sklearn.model_selection import train_test_split
import scvi

In [ ]:
control_key = "is_control"
condition_keys = "guide_merged"
condition_rep_keys = "gene_embeddings"
random_seed = 42
dataset_name = "Norman_hvg"
#sample_rep = "X_pca" 
sample_rep = "X_scVI" 
#sample_rep = "X_flatvi"
#sample_rep = "X_state"

In [ ]:
if sample_rep == "X_state":
    filePath = './data/raw/adata_Training_state_emb.h5ad'
    if not os.path.exists("./data/raw/adata_Training_state_emb.h5ad"):
        !state emb transform --model-folder ./data/SE-600M --input ./data/raw/adata_Training.h5ad --output ./data/raw/adata_Training_state_emb.h5ad
else:
    filePath = './data/raw/my_norman.h5ad'
adata = sc.read_h5ad(filePath)
#adata = adata[adata.obs.sample(frac=0.1, random_state=42).index].to_memory()
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

hvg = True
if hvg:
    sc.pp.highly_variable_genes(adata,n_top_genes=5000, subset=False)
    conditions = [(c.split('+')[0], c.split('+')[1]) for c in adata.obs['guide_merged'] if '+' in c]
    conditions = [item for sublist in conditions for item in sublist]
    genes_to_keep = np.unique(conditions)
    adata.var['highly_variable'] = adata.var['highly_variable'] + adata.var.gene_symbols.isin(genes_to_keep)
    adata = adata[:,adata.var['highly_variable'] == True].copy()
print(adata)

In [ ]:
adata.obs[control_key] = (adata.obs[condition_keys] == "ctrl")
gene_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

## splitting

In [ ]:
rng = np.random.default_rng(random_seed) 
test_ratio = 0.1
gene_list = list(gene_list)
zero_shot = False

if not zero_shot:
    # 分层抽样 先验证分布内学习能力
    pert_mask = adata.obs[control_key] == False
    y = adata.obs.loc[pert_mask, condition_keys].astype(str).values
    pert_indices = np.flatnonzero(pert_mask)

    train_idx, test_idx = train_test_split(
        pert_indices,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y 
    )
    adata_train = adata[train_idx].copy()
    adata_test = adata[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()
    adata_train.uns["normalized_m"] = 1 / (1-test_ratio)
    adata_test.uns["normalized_m"] = 1 / test_ratio
    adata_control.uns["normalized_m"] = 1
    print(gene_list)
else:
    # 按基因分割 zero-shot
    train_perts, test_perts, details = gene_pair_split(
        adata=adata,
        pert_key=condition_keys,
        rng=rng,
        test_ratio=test_ratio,
        combo_seen2_train_frac=0.75,
        make_canonical_col=True,      # 默认就是 True
        verbose=True,
        return_details=True,
    )
    pert_key = details["pert_key_used"]
    
    
    adata_control = adata[adata.obs[control_key] == True].copy()
    adata_train   = adata[adata.obs[pert_key].astype(str).isin(train_perts)].copy()
    adata_test    = adata[adata.obs[pert_key].astype(str).isin(test_perts)].copy()

In [ ]:
import pickle

with open("my_norman_simulation_1_0.75.pkl", "rb") as f:  
    data = pickle.load(f)


train_perts = [x for x in data["train"] + data["val"] if x != "ctrl"]
test_perts = [x for x in data["test"]]
adata_control = adata[adata.obs[control_key] == True].copy()
adata_train   = adata[adata.obs[condition_keys].astype(str).isin(train_perts)].copy()
adata_test    = adata[adata.obs[condition_keys].astype(str).isin(test_perts)].copy()

test_ratio = "gears"
zero_shot = True


In [ ]:
train_conditions = list(adata_train.obs[condition_keys].astype(str).unique())
test_conditions = list(adata_test.obs[condition_keys].astype(str).unique())

In [ ]:
from src.evaluate import classify_perturbations
classified_res, vocab = classify_perturbations(train_conditions, test_conditions,ctrl_tag='ctrl')

print(f"1. 单扰动 - Train中未出现 (New Single): {classified_res['single_new']}")
print(f"2. 单扰动 - Train中已出现 (Seen Single): {classified_res['single_seen']}")
print(f"3. 双扰动 - 0个Train基因 (Unseen Double): {classified_res['double_0']}")
print(f"4. 双扰动 - 1个Train基因 (1-Seen Double): {classified_res['double_1']}")
print(f"5. 双扰动 - 2个Train基因 (2-Seen Double): {classified_res['double_2']}")

In [ ]:
import pickle
import numpy as np

# for GEARS
def make_custom_split_pkl(train_perts, test_perts, out_path, seed=1, train_frac=0.9):
    # 1) 去重（可选但推荐，避免重复 condition）
    train_perts = list(dict.fromkeys(train_perts))
    test_perts  = list(dict.fromkeys(test_perts))

    # 2) 按比例切 train_perts -> train/val
    rng = np.random.default_rng(seed)
    idx = np.arange(len(train_perts))
    rng.shuffle(idx)

    n_train = int(len(train_perts) * train_frac)
    train_idx = idx[:n_train]
    val_idx   = idx[n_train:]

    train_list = [train_perts[i] for i in train_idx] + ["ctrl"]
    val_list   = [train_perts[i] for i in val_idx]
    test_list  = list(test_perts)

    # 3) 组成 set2conditions（这就是你代码里 pickle.load 需要读出来的东西）
    set2conditions = {
        "train": train_list,
        "val":   val_list,
        "test":  test_list,
    }

    # 4) 存 pkl
    with open(out_path, "wb") as f:
        pickle.dump(set2conditions, f)

    return set2conditions

# 用法示例：
split_dict = make_custom_split_pkl(train_conditions, test_conditions, "../../GEARS_misc/custom_split.pkl", seed=1, train_frac=0.9)

In [ ]:
del adata

## latent embedding

In [ ]:
n_comps = 64
n_hidden = 512
n_layers = 2
condition_rep_dict = pd.read_pickle("./data/processed/norman_gene_pert.pkl")
model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}"
flatvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}"
state_decoder_save_path = f"./data/processed/model/{sample_rep}_{dataset_name}"
model_save_path = None

load_embedding_model = True
if sample_rep in ["X_scVI"]:
    model_save_path = scvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{model_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{model_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{model_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep == "X_flatvi":
    model_save_path = flatvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{flatvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
elif sample_rep=="X_state":
    from src.preprocessing import build_train_eval_loaders,NBDecoderTrainer,NBDecoder
    model_save_path = state_decoder_save_path
    z_dim = adata_control.obsm["X_state"].shape[1] # 2058
    n_genes = adata_control.n_vars
    decoder = NBDecoder(z_dim=z_dim, n_genes=n_genes, hidden=(1024,2048,4096), dropout=0.1)
    train_loader, val_loader = build_train_eval_loaders(
                                    adata_train=adata_control,
                                    adata_eval=adata_train,   
                                    count_layer="counts",
                                    emb_key="X_state",
                                    batch_size=256,
                                )
    trainer = NBDecoderTrainer(decoder, lr=1e-4, device="cuda", use_amp=True)
    trainer.fit(train_loader, val_loader=val_loader, epochs=50)
    trainer.save(f"{state_decoder_save_path}.pt")

In [ ]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    n_layers = n_layers,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    model_save_path = model_save_path,
    control_key = control_key,
    condition_keys = condition_keys,
    condition_rep_keys = condition_rep_keys,
    batch_key = "gemgroup",
    condition_rep_dict = condition_rep_dict,
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )
if sample_rep == "X_pca":
    sample_rep_scaled = sample_rep + "_scaled" # 额 别忘了
else:
    sample_rep_scaled = sample_rep

In [ ]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep_scaled}_{n_comps}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [ ]:
print(adata_control)
print(adata_train)
print(adata_test)

In [ ]:
denoised_df = model_ref.get_normalized_expression(adata_control, return_mean=True,library_size=1e4)
raw_adata = adata_control.copy()
sc.pp.normalize_total(raw_adata, target_sum=1e4)
raw_matrix = raw_adata.X
if hasattr(raw_matrix, "toarray"):
    raw_matrix = raw_matrix.toarray()

In [ ]:
# 计算原始数据和重建数据的基因均值
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
mean_raw = np.mean(raw_matrix, axis=0)
mean_recon = np.mean(denoised_df.values, axis=0)

# 计算相关性 (Pearson 或 Spearman)
corr, _ = pearsonr(mean_raw, mean_recon)
print(f"Gene Mean Correlation (Raw vs Recon): {corr:.4f}")

# 可视化
plt.figure(figsize=(6, 6))
plt.scatter(mean_raw, mean_recon, s=1, alpha=0.5)
plt.plot([0, max(mean_raw)], [0, max(mean_raw)], 'r--') # 对角线
plt.xlabel("Raw Mean Expression (Normalized)")
plt.ylabel("Reconstructed Mean Expression")
plt.title(f"Reconstruction Quality (R = {corr:.2f})")
plt.xscale('log')
plt.yscale('log')
plt.show()